# Disaster Recovery Cost Prediction and Resilience Optimization
## Week 2 Day 3 — Model Training Pipeline

## Goal of this notebook

The purpose of this notebook is to train and compare three regression models for disaster recovery cost prediction using the processed disaster-level dataset.

The workflow includes:
- loading the processed data
- defining numeric and categorical features
- building a preprocessing pipeline
- training three regression models
- evaluating them with 5-fold cross-validation
- selecting the best model using mean CV R²
- logging results to MLflow
- saving the best trained model for later use

In [7]:
df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "processed_disasters.csv")

print("Total rows:", len(df))
print("Unique disasters:", df["disasterNumber"].nunique())

Total rows: 5163
Unique disasters: 5163


In [6]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.models.train import run_training_pipeline

results_df, best_model_name = run_training_pipeline()
display(results_df)
print("Best model:", best_model_name)

2026-04-09 08:22:23,447 | INFO | src.models.train | Loaded processed dataset | shape=(5163, 43)
2026-04-09 08:22:23,453 | INFO | src.models.train | Numeric features (11): ['declaration_year', 'declaration_month', 'incident_duration_days', 'state_5yr_disaster_count', 'high_cost_incident', 'fyDeclared', 'tribalRequest', 'fipsStateCode', 'fipsCountyCode', 'placeCode', 'region']
2026-04-09 08:22:23,454 | INFO | src.models.train | Categorical features (5): ['state', 'incidentType', 'declarationType', 'season', 'census_region']
2026-04-09 08:22:23,491 | INFO | src.models.train | Evaluating model: linear_regression
2026-04-09 08:22:26,785 | INFO | src.models.train | Evaluating model: random_forest
2026-04-09 08:22:34,257 | INFO | src.models.train | Evaluating model: xgboost
2026-04-09 08:22:38,269 | INFO | src.models.train | Cross-validation results:
       model_name  mean_cv_r2  std_cv_r2  mean_cv_rmse  std_cv_rmse  mean_cv_mae  std_cv_mae
    random_forest    0.856636   0.020837      3.361

,model_name,mean_cv_r2,std_cv_r2,mean_cv_rmse,std_cv_rmse,mean_cv_mae,std_cv_mae
0,random_forest,0.856636,0.020837,3.361276,0.244956,1.334235,0.075905
1,xgboost,0.832452,0.019835,3.637821,0.197939,1.645195,0.062949
2,linear_regression,0.616024,0.014058,5.517635,0.083146,4.203186,0.089804


Best model: random_forest


In [8]:
best_model = results_df.loc[results_df["mean_cv_r2"].idxmax(), "model_name"]
print(best_model)

random_forest


### Previous model result
During the development of the disaster recovery cost prediction pipeline, several critical challenges emerged that required careful investigation and iterative problem-solving to ensure the model was both valid and production-ready. One of the most significant issues was data leakage, which initially led to unrealistically high model performance (R² ≈ 0.99). 

In [2]:
results_df

,model_name,mean_cv_r2,std_cv_r2,mean_cv_rmse,std_cv_rmse,mean_cv_mae,std_cv_mae
0,random_forest,0.997338,0.000508,0.445696,0.042158,0.033497,0.003628
1,xgboost,0.996762,0.000866,0.489836,0.061161,0.134237,0.006534
2,linear_regression,0.573792,0.006340,5.664643,0.039855,4.594137,0.043344


In [3]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "processed_disasters.csv")

correlations = df.corr(numeric_only=True)["log_total_obligated_amount"].sort_values(ascending=False)

print(correlations.head(10))

log_total_obligated_amount    1.000000
hmProgramDeclared             0.754712
project_count                 0.459539
ihProgramDeclared             0.454618
incidentId                    0.453429
declaration_year              0.445204
fyDeclared                    0.444264
disasterNumber                0.387976
avg_project_amount            0.300471
totalObligatedAmountCatAb     0.252484
Name: log_total_obligated_amount, dtype: float64
